## Rice Upregulated Gene Group (3Di + AA Goto-Smith-Waterman alignment method result)

- Analysis using protein sequences used for structure prediction in the AlphaFold Structure database
- [alphafold/sequences.fasta](https://ftp.ebi.ac.uk/pub/databases/alphafold/sequences.fasta)

&nbsp;

- Integrate rice gene ID and gene expression indicator HN-score information with Foldseek results

In [1]:
import datetime
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from upsetplot import UpSet, from_contents
from matplotlib.patches import Patch
from IPython.display import display

In [2]:
rice_idmapping_file_path = "../Data/12_rice_uniprot_idmapping/rice_up_idmapping_all.tsv"
rice_hnscore_file_path = "../Data/07_extract_gene/HN5_genelist_rice_2505/HN5_genes_up_rice.tsv"
foldseek_result_file_path = "../Data/14_foldseek_result/evalue01/foldseek_output_uniprot_rice_up_all_evalue01.tsv"
main_target_species = 9606
# output_tsv_path = "../Data/14_foldseek_result/foldseek_output_uniprot_rice_up_9606_modified.tsv"

In [3]:
# re-execution for doctor thesis
VERSION = datetime.datetime.now().strftime("%Y%m%d")
print(f"Execute Date: {VERSION}")

Execute Date: 20260924


In [4]:
rice_idmapping = pl.read_csv(
    rice_idmapping_file_path,
    separator='\t'
)

rice_hnscore = pl.read_csv(
    rice_hnscore_file_path,
    separator='\t'
).select(
    "From",
    "HN5"
)

rice_idmapping_hnscore = rice_hnscore.join(
    rice_idmapping,
    on="From",
    how="left",
    coalesce=True,
    validate="1:m"
).sort(
    by=["HN5"],
    descending=[True]
)

# display(rice_idmapping)
display(rice_idmapping_hnscore)

From,HN5,UniProt Accession
str,i64,str
"""Os04g0107900""",255,"""A0A0P0W5Z1"""
"""Os04g0107900""",255,"""A0A0P0W604"""
"""Os04g0107900""",255,"""A0A0P0W643"""
"""Os04g0107900""",255,"""A0A0P0W6G9"""
"""Os04g0107900""",255,"""C7J151"""
…,…,…
"""Os06g0651200""",42,"""Q0DAI2"""
"""Os08g0374600""",42,"""A0A0P0XF07"""
"""Os10g0440500""",42,"""A0A0P0XV47"""


In [5]:
foldseek_result_all = pl.read_csv(
    foldseek_result_file_path,
    separator='\t'
).rename(
    {
        "query" : "UniProt Accession",
        "target" : "foldseek hit"
    }
).with_columns(
    (pl.col("UniProt Accession").str.extract(r"AF-(.*?)-F1", 1).alias("UniProt Accession")), # "?" means non-greedy
    (pl.col("foldseek hit").str.extract(r"AF-(.*?)-F1", 1).alias("foldseek hit"))
)

foldseek_result_all_v2 = rice_idmapping_hnscore.join(   
    foldseek_result_all,
    on="UniProt Accession",
    how="left",
    coalesce=True,
    validate="1:m"
)

foldseek_result_all_hit = foldseek_result_all_v2.filter(
    pl.col("foldseek hit").is_not_null()
)

display(foldseek_result_all_hit)

From,HN5,UniProt Accession,foldseek hit,evalue,prob,gapopen,pident,fident,nident,qstart,qend,qlen,tstart,tend,tlen,alnlen,qcov,tcov,lddt,qtmscore,ttmscore,alntmscore,rmsd,taxid,taxname,taxlineage,qaln,taln,mismatch,lddtfull
str,i64,str,str,f64,f64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,i64,str,str,str,str,i64,str
"""Os04g0107900""",255,"""A0A0P0W5Z1""","""A0A0E0QSW2""",5.5430e-11,1.0,2,79.2,0.792,80,1,101,101,6,103,103,101,1.0,0.951,0.8879,0.8004,0.7857,0.7857,3.434,4529,"""Oryza rufipogon""","""-_cellular organisms;d_Eukaryo…","""SSKKTMEINPENGIMEELRKRADADANDKS…","""SSKKTMEINPENAIMDELRKRADADKNDKS…",18,"""0.800,0.950,0.800,0.900,0.910,…"
"""Os04g0107900""",255,"""A0A0P0W5Z1""","""A0A0E0P4F7""",1.0260e-11,1.0,0,100.0,1.0,101,1,101,101,603,703,703,101,1.0,0.144,0.8301,0.7323,0.1175,0.1175,8.389,4529,"""Oryza rufipogon""","""-_cellular organisms;d_Eukaryo…","""SSKKTMEINPENGIMEELRKRADADANDKS…","""SSKKTMEINPENGIMEELRKRADADANDKS…",0,"""0.775,0.900,0.675,0.870,0.870,…"
"""Os04g0107900""",255,"""A0A0P0W5Z1""","""A0A178U8T0""",3.6370e-9,1.0,2,83.3,0.833,85,1,101,101,600,699,699,102,1.0,0.143,0.8303,0.755,0.1186,0.1186,8.34,3702,"""Arabidopsis thaliana""","""-_cellular organisms;d_Eukaryo…","""SSKKTMEINPENGIMEELRKRADADANDKS…","""SSKKTMEINPENSIMDELRKRADADKNDKS…",14,"""0.725,0.900,0.713,0.880,0.870,…"
"""Os04g0107900""",255,"""A0A0P0W5Z1""","""Q922F6""",6.2400e-9,1.0,1,83.1,0.831,84,1,101,101,393,491,491,101,1.0,0.202,0.8392,0.7336,0.1647,0.1647,7.342,10090,"""Mus musculus""","""-_cellular organisms;d_Eukaryo…","""SSKKTMEINPENGIMEELRKRADADANDKS…","""SSKKTMEINPENPIMEELRKRADADKNDKS…",15,"""0.800,0.925,0.725,0.880,0.860,…"
"""Os04g0107900""",255,"""A0A0P0W5Z1""","""A0A0E0QSW0""",5.8330e-9,1.0,2,80.1,0.801,81,1,101,101,411,508,508,101,1.0,0.193,0.817,0.7505,0.1609,0.1609,8.963,4529,"""Oryza rufipogon""","""-_cellular organisms;d_Eukaryo…","""SSKKTMEINPENGIMEELRKRADADANDKS…","""SSKKTMEINPENSIMDELRKRADADKNDKS…",17,"""0.775,0.900,0.675,0.880,0.870,…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Os10g0440500""",42,"""Q0IXG0""","""B4FCZ7""",0.02245,0.984,3,64.0,0.64,96,1,130,130,54,202,359,150,1.0,0.415,0.5899,0.335,0.1274,0.1274,23.01,4577,"""Zea mays""","""-_cellular organisms;d_Eukaryo…","""GVETVVTDTAGNKVVVTGAADAAELKERIE…","""GVESVTPDMAAGKVVVTGPADAVELKERIE…",33,"""0.439,0.506,0.432,0.406,0.447,…"
"""Os10g0440500""",42,"""Q0IXG0""","""A0A317Y153""",0.05181,0.961,5,59.3,0.593,92,1,130,130,54,203,375,155,1.0,0.4,0.5834,0.336,0.1226,0.1226,25.47,4577,"""Zea mays""","""-_cellular organisms;d_Eukaryo…","""GVETVVTDTAGNKVVVTGAADAAELKERIE…","""GVESVTPDMAAGKVVVTGPADAVELKERIE…",33,"""0.432,0.500,0.439,0.406,0.447,…"
"""Os10g0440500""",42,"""Q0IXG0""","""A0A3L6EK86""",0.03756,0.961,5,63.4,0.634,99,1,130,130,55,208,379,156,1.0,0.406,0.554,0.3349,0.1212,0.1212,25.12,4577,"""Zea mays""","""-_cellular organisms;d_Eukaryo…","""GVETVVTDTAGNKVVVTGAADAAELKERIE…","""GVESVTPDMAAGKVVVTGPADAVQLKERIE…",29,"""0.470,0.511,0.439,0.411,0.461,…"


&nbsp;

&nbsp;

&nbsp;

## No Hit Genes in All Set Species (Homo sapiens, Mus musculus, Arabidopsis thaliana, Zea mays, Oryza rufipogon)

In [ ]:
no_hit = foldseek_result_all_v2.filter(
    pl.col("foldseek hit").is_null()
).select(
    ["From", "UniProt Accession"]
).with_columns(
    pl.col("From").is_in(foldseek_result_all_hit["From"]).alias("exists_in_hit")
)

all_no_hit = no_hit.filter(
    pl.col("exists_in_hit") == False
).sort(
    ["From"], descending=False
)

print(all_no_hit.group_by("From").n_unique())
display(all_no_hit)

shape: (76, 3)
┌──────────────┬───────────────────┬───────────────┐
│ From         ┆ UniProt Accession ┆ exists_in_hit │
│ ---          ┆ ---               ┆ ---           │
│ str          ┆ u32               ┆ u32           │
╞══════════════╪═══════════════════╪═══════════════╡
│ Os01g0136050 ┆ 1                 ┆ 1             │
│ Os01g0184050 ┆ 1                 ┆ 1             │
│ Os01g0386500 ┆ 1                 ┆ 1             │
│ Os01g0715201 ┆ 1                 ┆ 1             │
│ Os01g0719125 ┆ 1                 ┆ 1             │
│ …            ┆ …                 ┆ …             │
│ Os11g0707100 ┆ 1                 ┆ 1             │
│ Os12g0111700 ┆ 1                 ┆ 1             │
│ Os12g0255850 ┆ 2                 ┆ 1             │
│ Os12g0418600 ┆ 1                 ┆ 1             │
│ Os12g0440300 ┆ 1                 ┆ 1             │
└──────────────┴───────────────────┴───────────────┘


/tmp/ipykernel_37222/717917391.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  ).with_columns(


From,UniProt Accession,exists_in_hit
str,str,bool
"""Os01g0136050""","""A0A0P0UXN9""",false
"""Os01g0184050""","""A0A0N7KCG6""",false
"""Os01g0386500""","""A0A0P0V2U1""",false
"""Os01g0715201""","""A0A0P0V7E4""",false
"""Os01g0719125""","""A0A0P0V7I7""",false
…,…,…
"""Os12g0111700""","""A0A0P0Y665""",false
"""Os12g0255850""","""A0A0P0Y8Y1""",false
"""Os12g0255850""","""C7J9X1""",false


In [15]:
HN5_up_info = pl.read_csv(
    "../Data/07_extract_gene/HN5_genelist_rice_2507/HN5_genes_up_rice_with_info.tsv",
    separator='\t',
    schema_overrides={
        "Oryzabase Trait Gene ID": pl.String
    }
).rename(
    {
        "GENEID": "From"
    }
)

all_no_hit_with_info = all_no_hit.join(
    HN5_up_info,
    on="From",
    how="left",
    coalesce=True,
    validate="m:m"
).sort(
    ["HN5"], descending=True
)

all_no_hit_with_info.write_csv(
    "../out/doctor_thesis/rice_up_all_no_hit_with_info.tsv",
    separator='\t'
)

display(all_no_hit_with_info)

From,UniProt Accession,exists_in_hit,UP5,DOWN5,UNCHANGED5,HN5,UPcount+1 / DOWNcount+1 5,Transcript_ID,Description,RAP-DB Gene Symbol Synonym(s),RAP-DB Gene Name Synonym(s),CGSNL Gene Symbol,CGSNL Gene Name,Oryzabase Gene Symbol Synonym(s),Oryzabase Gene Name Synonym(s),GO,InterPro,Transcript evidence,ORF evidence,Curation Date,Literature PMID,Oryzabase Trait Gene ID,Subcellular location,Comment,B5toI1
str,str,bool,i64,i64,i64,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Os01g0136050""","""A0A0P0UXN9""",false,232,10,118,222,21.181818,"""Os01t0136050-00""","""Similar to 16.9 kDa heat shock…",null,null,null,null,null,null,null,null,"""BT016978""","""E5D3J8 (UniProt)""",null,null,null,null,null,null
"""Os04g0108101""","""A0A0N7KIG5""",false,205,4,151,201,41.2,"""Os04t0108101-00""","""Hypothetical protein.""",null,null,null,null,null,null,null,null,"""BT016887""","""longestORF""",null,null,null,null,null,null
"""Os01g0184050""","""A0A0N7KCG6""",false,202,4,154,198,40.6,"""Os01t0184050-00""","""Hypothetical protein.""",null,null,null,null,null,null,null,null,"""BT016978""","""longestORF""",null,null,null,null,null,"""Exon-intron structures, exon a…"
"""Os04g0578550""","""A0A0P0WDU2""",false,168,9,183,159,16.9,"""Os04t0578550-00""","""Hypothetical conserved gene.""",null,null,null,null,null,null,null,null,"""CU405845""","""A2XWQ0 (UniProt)""",null,null,null,null,null,null
"""Os05g0296800""","""A0A0P0WK90""",false,162,3,195,159,40.75,"""Os05t0296800-01""","""Similar to Retrotransposon Kar…",null,null,null,null,null,null,null,null,"""AK108441""","""Q8H993 (UniProt)""",null,null,null,null,null,"""CDS sequences are different be…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Os01g0715201""","""A0A0P0V7E4""",false,61,19,280,42,3.1,"""Os01t0715201-01""","""Hypothetical conserved gene.""",null,null,null,null,null,null,null,null,"""AK288648""","""Q6ZHL7 (UniProt)""",null,null,null,null,null,null
"""Os03g0296200""","""A0A0P0VWE6""",false,54,12,294,42,4.230769,"""Os03t0296200-00""","""Conserved hypothetical protein…",null,null,null,null,null,null,null,null,"""tplb0003o16 (Wheat FLcDNA)""","""NP_001173381.1 (RefSeq)""",null,null,null,null,null,"""Exon-intron structures, exon a…"
"""Os06g0651200""","""A0A0N7KMI4""",false,64,22,274,42,2.826087,"""Os06t0651200-01""","""Conserved hypothetical protein…",null,null,null,null,null,null,null,null,"""AK107748""","""A2YFP3 (UniProt)""",null,null,null,null,null,null


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# Foldseek result (Human)

In [ ]:
foldseek_result_human = foldseek_result_all_hit.filter(
    pl.col("taxid").is_in([main_target_species])
)

display(foldseek_result_human)

In [ ]:
# Extract Each UniProt ID for Retrieving InterPro domain information via UniProt id mapping
# rice_up_rice_uniprot_idlist = foldseek_result_human.select(
#     "UniProt Accession"
# ).unique().write_csv(
#     "../Data/18_foldseek_hit_uniprot_idmapping/uniprot_id_rice_up_rice_idlist_for_interpro.txt",
#     separator="\t",
#     include_header=False
# )

# rice_up_human_uniprot_idlist = foldseek_result_human.select(
#     "foldseek hit"
# ).unique().write_csv(
#     "../Data/18_foldseek_hit_uniprot_idmapping/uniprot_id_rice_up_human_idlist_for_interpro.txt",
#     separator="\t",
#     include_header=False
# )

In [ ]:
# Calculate hit count (gene level)
hit_count_human = foldseek_result_human.group_by("From").agg(
        (pl.col("foldseek hit").count().alias("hit count (gene level)"))
    ).sort(
        ["hit count (gene level)"], descending=True
    )

display(hit_count_human)

In [ ]:
# foldseek_result_human.write_csv(
#     output_tsv_path,
#     separator='\t'
# )

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# Foldseek result (Mouse)

In [ ]:
foldseek_result_mouse = foldseek_result_all_hit.filter(
    pl.col("taxid").is_in([10090])
)

display(foldseek_result_mouse)

In [ ]:
# Calculate hit count (gene level)
hit_count_mouse = foldseek_result_mouse.group_by("From").agg(
        (pl.col("foldseek hit").count().alias("hit count (gene level)"))
    ).sort(
        ["hit count (gene level)"], descending=True
    )

display(hit_count_mouse)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;



# Foldseek result (Arabidopsis)

In [ ]:
foldseek_result_arabidopsis = foldseek_result_all_hit.filter(
    pl.col("taxid").is_in([3702])
)

display(foldseek_result_arabidopsis)

In [ ]:
hit_count_arabidopsis = foldseek_result_arabidopsis.group_by("From").agg(
    (pl.col("foldseek hit").count().alias("hit count (gene level)"))
).sort(
    ["hit count (gene level)"], descending=True
)

display(hit_count_arabidopsis)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;



# Foldseek result (Maize)

In [ ]:
foldseek_result_maize = foldseek_result_all_hit.filter(
    pl.col("taxid").is_in([4577])
)

display(foldseek_result_maize)

In [ ]:
hit_count_count_maize = foldseek_result_maize.group_by("From").agg(
    (pl.col("foldseek hit").count().alias("hit count (gene level)"))
).sort(
    ["hit count (gene level)"], descending=True
)

display(hit_count_count_maize)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;



# Foldseek result (Oryza rufipogon)

In [ ]:
foldseek_result_oryza_rufipogon = foldseek_result_all_hit.filter(
    pl.col("taxid").is_in([4529])
)

display(foldseek_result_oryza_rufipogon)

In [ ]:
hit_count_oryza_rufipogon = foldseek_result_oryza_rufipogon.group_by("From").agg(
    (pl.col("foldseek hit").count().alias("hit count (gene level)"))
).sort(
    ["hit count (gene level)"], descending=True
)

display(hit_count_oryza_rufipogon)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Upset plot (Gene Level Count)

In [ ]:
species_contents = {
    'Homo sapiens': hit_count_human["From"].to_list(),
    'Mus musculus': hit_count_mouse["From"].to_list(),
    'Arabidopsis thaliana': hit_count_arabidopsis["From"].to_list(),
    'Zea mays': hit_count_count_maize["From"].to_list(),
    'Oryza rufipogon': hit_count_oryza_rufipogon["From"].to_list(),
    'rice all no hit genes': all_no_hit["From"].unique().to_list(),
    'rice all genes': rice_idmapping["From"].unique().to_list()
}

upset_data = from_contents(species_contents)
display(upset_data)

In [ ]:
upset_data.to_csv(
    "../Data/14_foldseek_result_upset/rice_up_all_upset_data.tsv",
    sep="\t",
    index=True
)

In [ ]:
upset_plot = UpSet(
    upset_data,
    orientation='horizontal',
    show_counts="{:d}",
    subset_size='count',
    include_empty_subsets = False
)

upset_plot.style_subsets(present="Homo sapiens", 
                         facecolor="navy"
                         )

upset_plot.style_subsets(absent=["Homo sapiens", "Mus musculus"], 
                         facecolor="forestgreen"
                         )

upset_plot.style_subsets(absent=["Homo sapiens", "Mus musculus", "Arabidopsis thaliana", "Zea mays", "Oryza rufipogon"], 
                         facecolor="red"
                         )

fig = plt.figure(figsize=(14, 8), dpi=500)
axes = upset_plot.plot(fig=fig)

# Italicize binomial names; leave non-scientific labels upright
non_scientific = {"rice all genes", "rice all no hit genes"}
for tick in axes["matrix"].get_yticklabels():
    if tick.get_text() not in non_scientific:
        tick.set_fontstyle("italic")

plt.show()

In [ ]:
# Create legend independently
legend_elements = [
    Patch(facecolor="navy", label="including human hit"),
    Patch(facecolor='forestgreen', label='only plant hit')
]

fig_leg = plt.figure(figsize=(1.5, 0.5), dpi=500)
ax_leg = fig_leg.add_subplot(111)
ax_leg.legend(handles=legend_elements, loc='center')
ax_leg.axis('off') 
plt.show()
# fig_leg.savefig('legend.png', bbox_inches='tight')